# Laboratorio 7 - Spark MLlib
CC3066 - Data Science  

## Análisis exploratorio avanzado y segmentación

### Integrantes

- María José Girón Isidro
- Leonardo Dufrey Mejía Mejía
- Mia Alejandra Fuentes Mérida

---

## Objetivo

En este notebook se realiza la preparación, análisis exploratorio y
segmentación de los datos de Personas de la Encuesta Nacional de Empleo
e Ingresos Continua (ENEIC).

Para el desarrollo se utilizan los cuatro trimestres correspondientes
al año 2025. El primer trimestre de 2026 se mantiene separado para la
evaluación final.

In [25]:
import os
import math
import pandas as pd
import numpy as np
import openpyxl
import gc

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import matplotlib.pyplot as plt

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Laboratorio7-ENEIC")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 22:08:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versión de Spark: 3.5.1


In [4]:
print("Estoy trabajando en:")
print(os.getcwd())
print("Archivos disponibles:")
print(os.listdir("."))
print(os.listdir("./data"))

Estoy trabajando en:
/opt/app/CC3084-Laboratorio7
Archivos disponibles:
['data', 'Laboratorio7.ipynb', '.ipynb_checkpoints']
['Personas_ENEIC_T1_2025.xlsx', 'diccionarios', '.ipynb_checkpoints', 'Personas-ENEIC-T2-2025.xlsx', 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 'Base-de-datos-Personas-ENEIC-I-2026.xlsx', 'Base-de-datos-Personas-ENEIC-III-2025.xlsx']


In [5]:
DATA_DIR = "./data"

ARCHIVOS = {
    "2025T1": {
        "ruta": f"{DATA_DIR}/Personas_ENEIC_T1_2025.xlsx",
        "anio": 2025,
        "trimestre": 1
    },
    "2025T2": {
        "ruta": f"{DATA_DIR}/Personas-ENEIC-T2-2025.xlsx",
        "anio": 2025,
        "trimestre": 2
    },
    "2025T3": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-III-2025.xlsx",
        "anio": 2025,
        "trimestre": 3
    },
    "2025T4": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
        "anio": 2025,
        "trimestre": 4
    },
    "2026T1": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-I-2026.xlsx",
        "anio": 2026,
        "trimestre": 1
    }
}

for periodo, info in ARCHIVOS.items():
    print(periodo, "->", os.path.exists(info["ruta"]))

2025T1 -> True
2025T2 -> True
2025T3 -> True
2025T4 -> True
2026T1 -> True


In [6]:
COLUMNAS_REQUERIDAS = [
    "P05D01",       # salario mensual
    "P02A03",       # edad
    "P05C07A",      # antigüedad años
    "P05C07B",      # antigüedad meses
    "P05H01A",      # horas semanales
    "P03A03A",      # nivel educativo
    "P05C16",       # categoría ocupacional
    "DOMINIO",      # dominio
    "OCUPADOS",     # condición de ocupado
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

RENOMBRES = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado"
}